# DL Masterclass 03: Weight Initialization Theory

In this module, we explore the "criticality" of a neural network: its state at step zero.

A network's initial weights dictate whether gradients flow like a river or evaporate like mist. Before we start learning, we must ensuring the network is at a point where it is *capable* of receiving updates from the first batch of data.

---

## Table of Contents
1. **Symmetry Breaking:** Why identical weights guarantee failure.
2. **The Variance Problem:** Scaling laws for deep MLP forward passes.
3. **Xavier & He Initialization:** The industry standards for Tanh and ReLU.
4. **Orthogonal Initialization:** Solving recurrence stability for RNNs/LSTMs.
5. **Spectral Normalization:** Lipschitz constraints for GAN stability.
6. **The BatchNorm Interaction:** Why initialization still matters with normalization.

---

## 🎓 Socratic Deep Check
**The Zero-Gradient Dilemma:**
> *"If we initialize weights to be too small, the activations vanish. If we initialize them too large, the activations explode. But if we initialize them 'just right' such that the variance stays 1.0, are we guaranteed that the GRADIENTS will also stay stable during backpropagation? Or is there a separate condition for backward flow?"*

---
📈 **Production Signal:** Training failures in LLMs are often traced back to poor initialization of the Transformer's FFN layers or improperly scaled Residual paths.

---


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Initialization is the first hyperparameter you tune. For CNNs, He initialization is the default. For Transformers, we often use truncated normal distributions with small standard deviations like 0.02. For GANs, we use Spectral Normalization to prevent the discriminator from becoming too powerful too quickly.

**Mental Model:** Think of initialization as setting the "equilibrium point" of a system. 
- **Xavier/He** keeps the variance of signal flow constant across layers.
- **Orthogonal** ensures a vector's length stays the same after being multiplied by a matrix 100 times (crucial for time sequences).
- **Spectral** caps the maximum "stretching" a matrix can do to any input vector.

**Common Junior Pitfall:** Believing that Batch Normalization makes initialization irrelevant. While BN helps stability, poor initialization leads to slower convergence and can trap the model in bad local minima early on. Always start with mathematically sound initialization (Xavier/He).

---


## 1. Symmetry Breaking

A network of 1,000 neurons initialized with all zeros behaves exactly like a network with 1 neuron. 

Each neuron $h_i = \sigma(\sum_j w_{ij}x_j)$. If all $w_{ij}$ are the same (e.g. 0.1), then $h_1 = h_2 = \dots = h_{1000}$. During backpropgation, since they produce the same error, they receive the **exact same gradient update**. They stay identical copies forever. We use random initialization to **Break Symmetry**, allowing neurons to specialize in different features.

📈 **Production Signal:** Almost all libraries use `torch.nn.init.kaiming_uniform_` or similar by default, ensuring symmetry is broken with a distribution scaled to the layer size.


## 2. The Variance Problem (Explosion vs Collapse)

Consider a layer with $n$ inputs. Each output $y$ is computed as $y = \sum_{i=1}^n w_i x_i$. 
Assuming $x$ and $w$ are independent with mean 0:
\begin{aligned}
\text{Var}(y) &= \text{Var}(\sum w_i x_i) \\
&= \sum \text{Var}(w_i x_i) \\
&= \sum (E[x_i^2] E[w_i^2] - (E[x_i]E[w_i])^2) \\
&= \sum \text{Var}(x_i) \text{Var}(w_i) \quad \text{(using independence and mean 0)} \\
&= n \cdot \text{Var}(x) \cdot \text{Var}(w)
\end{aligned}

**The Law of Stability:** To keep the variance of $y$ equal to the variance of $x$ (so the signal doesn't explode or collapse), we need $n \cdot \text{Var}(w) = 1$.

Therefore: $\text{Var}(w) = \frac{1}{n}$. This is the foundation of modern initialization.

### 🎓 Socratic Deep Check
> *"If your weights have Var(w) = 1/n, your signal stays stable at 1.0. If you accidentally set it to 1.1/n, after 100 layers, how much larger has your signal grown? Does this qualify as an 'explosion'?"*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def simulate_signal_flow(std_scale, n_layers=50, n_nodes=500):
    x = np.random.randn(1, n_nodes)
    variances = [np.var(x)]
    
    for _ in range(n_layers):
        # Random weight matrix with specific scale
        w = np.random.randn(n_nodes, n_nodes) * std_scale
        x = x @ w
        variances.append(np.var(x))
        
    return variances

scale_small = 0.01
scale_large = 1.0
scale_perfect = np.sqrt(1 / 500)

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(simulate_signal_flow(scale_small), label="Too Small (0.01)", color='blue')
ax.plot(simulate_signal_flow(scale_perfect), label="Mathematically Perfect (Xavier Scale)", color='green')
ax.plot(simulate_signal_flow(0.06), label="Slightly Large (0.06)", color='orange') # 0.06 approx 1.34/sqrt(500)
ax.set_yscale('log')
ax.set_title("Log-Scale Signal Variance over 50 Layers")
ax.set_ylabel("Variance (log axis)")
ax.set_xlabel("Layer Depth")
ax.legend()
plt.show()

"""
What just happened?
We simulated a forward pass through 50 layers. On a log scale, even a tiny mismatch in variance
(orange line) leads to an exponential explosion. The green line represents the 'Critical State'
where the network stays playable all the way through.
"""

## 3. Xavier & He Initialization

Activation functions change the variance equation:
- **Xavier (for Tanh):** Tanh is linear near zero, so it follows the $1/n$ rule roughly. Xavier proposed $\text{Var}(w) = \frac{2}{n_{in} + n_{out}}$ to balance forward and backward signal stability.
- **He/Kaiming (for ReLU):** Since ReLU kills half the distribution ($\max(0, x)$), it cuts variance in half. To compensate, we double the input variance to **$2/n_{in}$**.

📈 **Production Signal:** Most deep learning frameworks default to **He Initialization** for almost all hidden layers unless specified otherwise.


## 4. Orthogonal Initialization — The Gold Standard for RNNs

In Recurrent Neural Networks (RNNs), the *same* matrix $W$ is multiplied repeatedly by the hidden state $h$ at every timestep. In a sequence of 200 tokens, $W$ is applied 200 times. 

If the largest eigenvalue of $W$ is 1.01, the signal explodes. If 0.99, it vanishes. We need $W$ to be an **Orthogonal Matrix**, where all eigenvalues have a magnitude of exactly 1.0.

### The Invariance Proof
If $W$ is orthogonal ($W^T W = I$):
1. $||Wx||_2^2 = (Wx)^T (Wx)$
2. $= x^T W^T W x$
3. Since $W^T W = I$: $x^T I x = x^T x = ||x||_2^2$

Mathematical Result: An orthogonal linear transformation preserves the L2-norm of the input vector. No growth, no shrinking.

### Implementation SVD Trick
To get an orthogonal matrix from a random one:
1. Generate random matrix $M$.
2. Perform Singular Value Decomposition: $M = U S V^T$.
3. $U$ and $V^T$ are orthogonal by definition. Return $U$.

### 🎓 Socratic Deep Check
> *"If an orthogonal matrix preserves vector norms perfectly, does it still allow the network to 'learn' by changing the weights? Or does the orthogonality constraint limit the network's capacity to fit the data?"*

In [ ]:
def orthogonal_init(shape):
    # SVD Trick for Orthogonality
    M = np.random.randn(*shape)
    u, s, vh = np.linalg.svd(M, full_matrices=False)
    return u if u.shape == shape else vh

def simulate_rnn(init_fn, timesteps=200, dim=256):
    h = np.random.randn(dim)
    w = init_fn((dim, dim))
    norms = [np.linalg.norm(h)]
    
    for _ in range(timesteps):
        h = np.tanh(h @ w) # Using Tanh to keep range bounded but illustrate stability
        norms.append(np.linalg.norm(h))
    return norms

norms_rand = simulate_rnn(lambda s: np.random.randn(*s) * 0.1)
norms_xavier = simulate_rnn(lambda s: np.random.randn(*s) * np.sqrt(1/s[0]))
norms_ortho = simulate_rnn(orthogonal_init)

plt.figure(figsize=(10, 5))
plt.plot(norms_rand, label="Random Std=0.1")
plt.plot(norms_xavier, label="Xavier Init")
plt.plot(norms_ortho, label="Orthogonal Init")
plt.title("Stability in Recurrent Chains (200 timesteps)")
plt.ylabel("L2 Norm of Hidden State")
plt.legend()
plt.show()

"""
What just happened?
In an RNN, Xavier is insufficient because it only balances variance for ONE step. 
Multiple steps amplify tiny errors. Notice the Orthogonal hidden state stays perfectly stable 
near its original norm, while others slowly decay or vanish.
"""

## 5. The Initialization-BatchNorm Interaction

Many believe that **Batch Normalization (BN)** makes initialization irrelevant because BN re-normalizes every layer. 

**Senior Insight:** BN provides stability for *divergent* initializations (preventing NaNs), but it does not fix *inefficient* ones. If you initialize with poorly scaled weights, the first many gradients pass through BN with high noise, leading to slower convergence.

📈 **Production Signal:** He + BN converges approximately **2x faster** than Random + BN in deep ResNets.

### 🎓 Socratic Deep Check
> *"If Batch Normalization calculates the mean and variance of a batch and shifts the data to zero-mean unit-variance, does it destroy the mathematical 'work' your careful Xavier/He initialization just did? Why or why not?"*

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

def train_model(use_he, use_bn, lr=0.01):
    layers = []
    for _ in range(20):
        layers.append(nn.Linear(100, 100))
        if use_bn: layers.append(nn.BatchNorm1d(100))
        layers.append(nn.ReLU())
    layers.append(nn.Linear(100, 10))
    model = nn.Sequential(*layers)
    
    # Apply manual initialization
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if use_he: nn.init.kaiming_normal_(m.weight)
            else: nn.init.normal_(m.weight, std=1.0) # Poor Choice
            
    optimizer = optim.SGD(model.parameters(), lr=lr)
    data = torch.randn(32, 100)
    target = torch.randint(0, 10, (32,))
    
    losses = []
    for _ in range(50):
        optimizer.zero_grad()
        loss = nn.CrossEntropyLoss()(model(data), target)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

loss_bad = train_model(False, False)
loss_bad_bn = train_model(False, True)
loss_he_bn = train_model(True, True)

plt.plot(loss_bad, label="Bad Init (N(0,1)) + No BN (Static Log-loss)", linestyle=':')
plt.plot(loss_bad_bn, label="Bad Init + BN")
plt.plot(loss_he_bn, label="He Init + BN")
plt.title("The Convergence Gap: 20-Layer MLP")
plt.ylabel("Loss")
plt.legend()
plt.show()

"""
What just happened?
Bad initialization without BN is impossible (loss never drops). 
Adding BN makes the bad initialization 'survivable', but notice that using He + BN 
still converges faster. Good initialization provides a better 'starting point' for 
the optimizer, even in normalized landscapes.
"""

## 6. Spectral Normalization for Discriminators

In Generative Adversarial Networks (GANs), the discriminator is often too powerful, causing the generator's gradients to explode or collapse. We stabilize this using **Spectral Normalization**.

The **Spectral Norm** $\sigma(W)$ is the largest singular value of the weight matrix $W$. By dividing $W$ by its spectral norm, we ensure that the matrix transformation is **1-Lipschitz**:
$$||W_{sn}x||_2 \le ||x||_2$$

### The Power Iteration Method
Calculating SVD in every forward pass is too slow. PyTorch uses "Power Iteration" to approximate the largest eigenvalue:
1. Pick random vector $u$.
2. Loop: $v = Wu / ||Wu||_2, u = W^T v / ||W^T v||_2$
3. $\sigma(W) \approx u^T W^T v$

📈 **Production Signal:** `torch.nn.utils.spectral_norm` is the standard tool for training stable BigGANs and StyleGAN discriminators.


In [ ]:
def power_iteration(W, steps=10):
    u = np.random.randn(W.shape[1], 1)
    for _ in range(steps):
        v = W @ u
        v /= np.linalg.norm(v)
        u = W.T @ v
        u /= np.linalg.norm(u)
    
    sigma = u.T @ W.T @ v
    return sigma.item()

W_test = np.random.randn(100, 100)
sigma_pi = power_iteration(W_test)
sigma_true = np.linalg.svd(W_test, compute_uv=False)[0]

print(f"Estimated Spectral Norm (Power Iteration): {sigma_pi:.6f}")
print(f"True Spectral Norm (SVD):               {sigma_true:.6f}")
print(f"Relative Error: {abs(sigma_pi - sigma_true)/sigma_true:.2%}")

"""
What just happened?
We implemented the Fast Spectral Norm approximation. Within 10 steps of power iteration,
we are usually within 0.1% of the true SVD value. This allows us to regularize matrix
scale in real-time during training.
"""

---
## ✅ Knowledge Check

### Q1: Why Orthogonal over Xavier for RNNs?
<details><summary>👉 Answer</summary>
Xavier initialization only ensures variance is held constant for a single layer pass. In RNNs, the same matrix is applied repeatedly. If its eigenvalues are even slightly above or below 1.0, the signal explodes or vanishes over long sequences. Orthogonal matrices have eigenvalues exactly on the unit circle (magnitude 1.0), ensuring perfect stability across hundreds of timesteps.
</details>

### Q2: Spectral Normalization vs Weight Decay
<details><summary>👉 Answer</summary>
Weight decay penalizes the L2-norm of all weights uniformly. Spectral Normalization specifically targets the direction of maximum 'stretching' (the largest singular value). This makes it much more effective for enforcing mathematical constraints like the Lipschitz-1 property needed for stable GAN training.
</details>


---
## 🔨 Practical Practice

### Tier 1: Beginner (Replicate)
1. Implement `nn.init.xavier_uniform_` from scratch and verify that the variance of the generated weights matches $\frac{2}{n_{in} + n_{out}}$.
2. Plot the singular value distribution of a 100x100 random Gaussian matrix before and after **Spectral Normalization**.

### Tier 2: Intermediate (Extend)
1. **The Skip Connection Advantage:** Build a 50-layer PRE-ACTIVATION ResNet. Show that it is significantly more robust to 'Bad' initialization (std=1.0) than a standard MLP. Plot the gradients at the input layer for both.
2. **Power Iteration Convergence:** Plot the error of the Power Iteration method vs. number of steps (1 to 50). How many steps do we truly need in production to get within 0.01% error?

### Tier 3: Advanced (Derive/Engineer)
1. **Modern Derivation Challenge:** Derive, from first principles, why He initialization requires a factor of 2 for ReLU but a factor of $2/(1 + \alpha^2)$ for LeakyReLU with slope $\alpha$. 
   *Hint: Start from $\text{Var}(y) = n \cdot E[w^2] E[x^2]$. Calculate $E[x^2]$ after activation by treating $x$ as $\mathcal{N}(0, \sigma^2)$ and integrating the activation function over the normal PDF. Account for the 'energy' passed by the negative slope $\alpha$.*
